# Reranker Search implementation

In [1]:
%load_ext autoreload
%autoreload 2

In [25]:
from dotenv import load_dotenv
import os

load_dotenv()

app_base_url = os.getenv('LOCATION', 'http://localhost:6333')

app_base_url

'http://localhost:6333'

In [26]:
from app.main import RerankingSearch


reranking_search = RerankingSearch(app_base_url)

In [27]:
reranking_search.ingest(force=True)

/Users/andreicristea/personal/qdrant-interview/app/dataset.py:42: DtypeWarning: Columns (0: latitude, 1: longitude) have mixed types. Specify dtype option on import or set low_memory=False.
  self.voyage_df = pd.read_csv(


In [36]:
from pathlib import Path


_ROOT_PATH = Path.cwd().parent 

In [58]:
import pandas as pd


search_queries = pd.read_csv(f'{_ROOT_PATH}/datasets/questions.csv')
search_queries

,query,specificity,facets,intent
0,sea and oceans,abstract,single,discover
1,romantic dinner with wine and great architecture,moderate,multi,discover
2,medieval history and ancient fortifications,moderate,single,discover
3,ancient ruins and archaeological sites,specific,single,discover
4,cheap hostel with wifi near the train station ...,specific,multi,explore
5,relaxing weekend escape with nature and good food,abstract,multi,discover
6,extreme sports and adrenaline,abstract,multi,discover
7,star gazing,abtract,single,discover
8,find for me peaceful place where I can party,abstract,multi,discover


In [40]:
import time
from app.models import SearchStrategies
from app.search import SEARCH_STRATEGIE_CLASSES, CitySearchParams

def create_search_result(query: str):
    search_results: list[tuple[str, SearchStrategies, float]]  = []
    t0 = time.perf_counter() 
    for strategy in SearchStrategies:
        if strategy == SearchStrategies.EXPLORATION_SEARCH:
            result = reranking_search.search(query, strategy, CitySearchParams(city='Kiel'))
        else:
            result  = reranking_search.search(query, strategy)
        
        elapsed = time.perf_counter() - t0
        str_result = SEARCH_STRATEGIE_CLASSES[strategy].to_readable_str(result)
        search_results.append((str_result, strategy, elapsed))
    
    for str_result, strategy, elapsed in search_results:                                                                                       
        print(f"\n=== {strategy.value.upper()} ({elapsed*1000:.1f}ms) ===")
        print(str_result) 

### Query 1: sea and oceans 

Really ambigious and abstract search query. Lets see how each strategy can handle it

In [44]:
sea_and_oceans = search_queries.at[0, "query"]
sea_and_oceans

'sea and oceans'

In [45]:

create_search_result(sea_and_oceans)


=== SIMPLE_SEARCH (136.3ms) ===
[score=0.8773] Fish at the Baltic sea in Kaliningrad (do): All year round
[score=0.8462] Ordu in Samsun (other): — on the way to more popular cities and sites of eternally rainy and green Eastern Black Sea
[score=0.8402] L’aparte in Baku (eat): European and Azeri Cuisine
[score=0.8363] Asia in Zagreb (eat): Chinese restaurant
[score=0.8304] Museum of the World Ocean in Kaliningrad (see): Includes two museum ships and one submarine.
[score=0.8292] El Cormorán in Santander (eat): dining by the sea
[score=0.8257] Dover Strait in Burgas (drink): Bar & cigars
[score=0.8252] Ivanovskiy Bridge in Astrakhan (other): Over the First of May canal
[score=0.8234] Bol d'Or in Geneva (do): Yacht Race (biggest in Europe).
[score=0.8233] Respublika Luks in Baku (eat): European and Azeri Cuisine
[score=0.8230] Murcia: You could be looking for:
[score=0.8228] Valladolid: You may be looking for:
[score=0.8219] Ukraine in Saint Petersburg (other): +7 812 331-5166
[score=0.8

### Query 2: Romance & others

Query 2 is a more specific request with concerete intent: to find nice romantic place. Moreover it also has multiple goals

In [ ]:
romantic_dinner: str = search_queries.at[1, "query"]
romantic_dinner

'romantic dinner with wine and great architecture'

In [48]:
create_search_result(romantic_dinner)


=== SIMPLE_SEARCH (99.2ms) ===
[score=0.8795] Blaue Ente in Zurich (eat): Romantic cuisine in a beautiful building.
[score=0.8737] Gelín in Santander (eat): Traditional and rustic
[score=0.8731] L’aparte in Baku (eat): European and Azeri Cuisine
[score=0.8714] Irish Pub in Uzhhorod (drink): Nice pizzas.
[score=0.8671] La Piazzetta in Cluj-Napoca (eat): A bit kitschy decor, but good pizza.
[score=0.8667] Varanda Da Barra in Porto (eat): Great restaurant that serves traditional Portuguese, Italian and "International" food. Nice riverside view.
[score=0.8661] Casa Lac in Zaragoza (eat): An excellent choice for higher-end tapas
[score=0.8651] In Vino Cafe in Perm (drink): Wine bar with food.
[score=0.8627] Srčeko in Zagreb (eat): A very romantic little restaurant.
[score=0.8621] Jardin moderne in Rennes (drink): Mostly a music hall, not a bar
[score=0.8615] Toscana in Baku (eat): Italian Cuisine.
[score=0.8602] Casablanca in Zurich (drink): Cool, modern setting.
[score=0.8593] Chateau in 

### Query 3: History & Fortifications 

Straighforward query that asks a question that can be understood without ambiguity

In [51]:
history_fort: str = search_queries.at[2, "query"]
history_fort

'medieval history and ancient fortifications'

In [52]:
create_search_result(history_fort)


=== SIMPLE_SEARCH (167.3ms) ===
[score=0.9053] Petersberg Citadel in Erfurt (see): a historical defence system and a great lookout over the city
[score=0.8834] Gavurkale and Kulhoyuk in Ankara (see): rock friezes and  Hittite burial grounds
[score=0.8833] Museo Civico Medievale in Bologna (see): Part of Musei Civici d'Arte Antica
[score=0.8789] Citadel in Ankara (see): There were laid by the Galatians on a prominent lava outcrop, and the rest was completed by the Romans. Walk through the cobbled streets lined by old houses to climb up to one of the towers, which offers a good view of the sprawling city below and the surrounding mountains.
[score=0.8767] Fortress walls and towers in Baku (see): Built in 11th–12th centuries
[score=0.8744] Bogazkale in Samsun (other): &mdash; town close to '''Hattuşaş''', which was once the capital of Hittite Empire, indigenous people of Anatolian highlands
[score=0.8713] Kasematten in Dresden (see): The remains of the old fort. Gives you a glimpse of wh

### Query 4: Ancient & Archeology

This query is very similar to previous one, however, it has one interesting property - it is converned with ancient landmarks, which is not the same as medieval. We can see how well each model can notice this distinction


In [53]:
ancient_fort: str = search_queries.at[3, "query"]
ancient_fort

'ancient ruins and archaeological sites'

In [54]:
create_search_result(ancient_fort)


=== SIMPLE_SEARCH (183.8ms) ===
[score=0.8913] Gavurkale and Kulhoyuk in Ankara (see): rock friezes and  Hittite burial grounds
[score=0.8691] Archaeological Museum in Ioannina (see): It presents many of the findings from the excavations at Dodona area
[score=0.8683] Archaeological Museum of Adana in Adana (see): Re-opened for visits after restoration (except for stone monuments section, still being restored).
[score=0.8670] Museo di Antropologia in Bologna (see): Bones, and artefacts of prehistoric Italians.
[score=0.8653] Gileyli mosque in Baku (see): built in 1309, rebuilt 1800's
[score=0.8615] Dmanisi archaeological site in Tbilisi (see): &mdash; a bronze age settlement and the 1.8 million year-old fossilized remains of the first human in Europe have been discovered here
[score=0.8613] Bogazkale in Samsun (other): &mdash; town close to '''Hattuşaş''', which was once the capital of Hittite Empire, indigenous people of Anatolian highlands
[score=0.8600] Gubkiv Castle in Rivne (see):

### Query 5: Very Concrete Request 

This query is very on point: it provides the place and also the what concretely user wants: cheap place, next to train


In [55]:
hotel_central_station: str = search_queries.at[4, "query"]
hotel_central_station

'cheap hostel with wifi near the train station in Kiel'

In [56]:
create_search_result(hotel_central_station)


=== SIMPLE_SEARCH (149.5ms) ===
[score=0.8915] Nicosia Youth Hostel in Nicosia (sleep): Free WIFI available.
[score=0.8902] Innsa Hostel in Valencia (sleep): Free Wi-Fi, has onsite bar/restaurant.
[score=0.8859] 2Go4 Hostel in Brussels (sleep): Very clean and very modern and chic. Free wi-fi (ask at reception for a code).
[score=0.8851] The White Tree Hostel in Pristina (sleep): Free internet access and Wi-Fi, Free cable TV in the lobby. (Taxi from the bus station shouldn't be more than €2 and from the airport about €15)
[score=0.8816] Peanuts Hostel in Kiel (sleep): Harriesstraße 2, 24114 Kiel
[score=0.8803] Casa España in Madrid (sleep): Cheap Breakfast: 2 Euro The nearest metro station is Plaza de España. Popular budget hostel in the city center. Free WiFi, linens included
[score=0.8803] Oranjin Hostel in Kazan (sleep): Closest hostel to train station. Guest kitchen, free WiFi, cosy common room.
[score=0.8802] Alex30 Hostel in Stuttgart (sleep): Central hostel with breakfast buffet

### Query 6: Relaxing weekend

This query is interesting from the point of view that it provides some time dependency 


In [60]:
relaxing_weekend: str = search_queries.at[5, "query"]
relaxing_weekend

'relaxing weekend escape with nature and good food'

In [61]:
create_search_result(relaxing_weekend)


=== SIMPLE_SEARCH (252.9ms) ===
[score=0.8818] Cafe Louf in Kiel (eat): Nice breakfast buffet on the weekend
[score=0.8730] L'Orangerie in Rouen (eat): Typical French cuisine. Nice setting on busy weekends.
[score=0.8709] PIVOLEND in Skopje (do): Gourmet weekend with beer.
[score=0.8708] Happy Valley in Rome (sleep): It has a pool, a bar, a restaurant and a minimarket.
[score=0.8669] La Piazzetta in Cluj-Napoca (eat): A bit kitschy decor, but good pizza.
[score=0.8667] Les Enfants Gâtés in Lyon (eat): Very good ice cream, on a lovely neighbourhood square. Also a good Sunday brunch.
[score=0.8658] Gelín in Santander (eat): Traditional and rustic
[score=0.8653] Pain et Cie in Lyon (eat): This place is quite popular for its Sunday brunch.
[score=0.8648] Green House in Novi Sad (eat): Vegetarian snacks, sandwiches, muffins, cakes. Vojvode Knicánina 1.
[score=0.8647] Reis'in Yeri in Trabzon (drink): Grill house and tea garden with a view on the old Genoese fortress. You might be able to ge

### Query 7: Singular query 

This query checks in general how for a single interest model match. This is diffent from `medieval history and ancient fortifications` because it asks a more nuanced question. However its not as abtract as `sea and oceans` which is very context-specific 

In [62]:
relaxing_weekend: str = search_queries.at[6, "query"]
relaxing_weekend

'extreme sports and adrenaline'

In [63]:
create_search_result(relaxing_weekend)


=== SIMPLE_SEARCH (208.7ms) ===
[score=0.8379] Bol d'Or in Geneva (do): Yacht Race (biggest in Europe).
[score=0.8375] Valladolid: You may be looking for:
[score=0.8368] Ice Climbing World Cup in Kirov (see): Annual competition.
[score=0.8362] Ironman Austria in Klagenfurt (do): Takes place every year with a large number of participants (2007: 2200 athletes).
[score=0.8352] Formula One in Budapest (do): Car racing.
[score=0.8345] Simpel Session in Tallinn (do): International skateboarding and BMX event.
[score=0.8311] Trebevic bobsled track in Sarajevo (see): Place where bobsled competition takes place during '84 Olympic Games. Partially destroyed during war. Amazing place for people who like to see ruins. This is also great place for risk takers - you can rent a bike in city and ride down in concrete bobsled path. Do not leave main paths - there can be still mines hidden in the forest!
[score=0.8304] Delfin Hotel in Baku (sleep): Situated 9&nbsp;km out of town. Opportunity to play pa

### Query 8: Star gazing 

A very abstract query which might be even not represented in the dataset

In [ ]:
star_gaze: str = search_queries.at[7, "query"]
star_gaze

'star gazing'

In [ ]:
create_search_result(star_gaze)


=== SIMPLE_SEARCH (163.3ms) ===
[score=0.8283] Fish at the Baltic sea in Kaliningrad (do): All year round
[score=0.8264] None in Arkhangelsk (do): Walk through Northern Dvina embankment, see kilometres of water and feel strong wind in any season.
[score=0.8239] Festival of lights in Berlin (do): Famous buildings are illuminated in a special way
[score=0.8237] Planetario de Madrid in Madrid (see): Features several exhibits related to space exploration, two screens playing documentaries, an interactive area and, of course, the planetarium. Projections last 45 minutes each. Different ones play on different days so check their website. Note that all the exhibits are explained in Spanish only and the projections in the planetarium are also in Spanish.
[score=0.8217] Praga in Uzhhorod (sleep): four star
[score=0.8201] Planetarium in Stuttgart (do): A fascinating astronomical journey, projected by optical hightech equipment: Carl Zeiss Planetarium . Note that almost all the shows conducted h

### Query 9: Contradictory query 

This query is interesting because users asks for contradictory features which might be even not detected on both dataset and purely model level.

In [67]:
contra: str = search_queries.at[8, "query"]
contra

'find for me peaceful place where I can party'

In [68]:
create_search_result(contra)


=== SIMPLE_SEARCH (182.0ms) ===
[score=0.8780] Soul Pub in Skopje (drink): Good music and beer.
[score=0.8701] club Soul in Novi Sad (drink): only electronic music, after parties & free entrance
[score=0.8699] Happy Valley in Rome (sleep): It has a pool, a bar, a restaurant and a minimarket.
[score=0.8677] Bárka in Debrecen (drink): In the cellar of the Town hall on main square is good for those who like rock, table football and dancing.
[score=0.8677] La Campana de los Perdidos in Zaragoza (drink): Enjoy a beer while listening live music, theatre, poetry from Wednesday to Sunday
[score=0.8665] BP Club in Zagreb (drink): Jazz and blues lovers should check it out.
[score=0.8660] Cafe "The Sting" in Novi Sad (drink): Vojvodina Stadium 458-155
[score=0.8651] Ryan's Irish Pub in Porto (drink): In the Ribeira, nice cozy atmosphere and friendly bar staff. Always a good place to start
[score=0.8638] Le You in Brussels (drink): For young clubbers who just want to party, 2 minutes walking due 

| Query                           | Spec.    | Facets | Intent   | Ranking                                          | Key Insight                                                                 |
|--------------------------------|----------|--------|----------|--------------------------------------------------|-----------------------------------------------------------------------------|
| sea and ocean                  | abstract | single | discover | top-down > bottom-up > encoder > simple           | city descriptions capture geography better than POIs                        |
| romantic dinner + architecture | moderate | multi  | discover | top-down > bottom-up > encoder > simple           | facets split naturally across hierarchy levels                              |
| medieval fortifications        | specific | single | discover | bottom-up ≈ encoder ≈ simple > top-down           | all methods work when vocabulary directly matches                           |
| ancient ruins & archaeology    | specific | single | discover | bottom-up > encoder > simple > top-down           | rich POI descriptions don't need city-level help                            |
| hostel wifi train in Kiel      | specific | multi  | explore  | top-down ≈ encoder > simple > bottom-up           | named city acts as hard constraint                                          |
| relaxing escape + nature + food| abstract | multi  | discover | all weak (bottom-up ≈ simple > top-down > encoder)| mood-based queries hit data vocabulary mismatch                             |
| extreme sports and adrenaline  | specific | single | discover | simple > bottom-up > top-down > encoder           | rare topic scattered across cities, grouping hurts                          |
| star gazing                    | abstract | single | discover | all fail (simple > others)                        | polysemy — "star" dominantly means hotel rating                             |
| peaceful place to party        | abstract | multi  | discover | bottom-up > simple > encoder > top-down           | ColBERT resolves contradiction via independent token matching               |